##### 下载短篇小说THE VERDICT作为文本数据

In [2]:
import urllib.request

url = ("https://raw.githubusercontent.com/rasbt/"
       "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
       "the-verdict.txt")
file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

('the-verdict.txt', <http.client.HTTPMessage at 0x1e035bff040>)

#### 代码清单2-1 通过Python读取短篇小说THE VERDICT作为文本样本

In [3]:
with open ("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


##### 设计的分词方法

In [4]:
import re

preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]  # 移除空白符
print("Total number of tokens:", len(preprocessed))

Total number of tokens: 4690


In [5]:
print(preprocessed[:30])  # 打印前30个标记

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


##### 创建词元列表，确定词汇表的大小

In [6]:
all_words = sorted(set(preprocessed))
print("Total number of unique words:", len(all_words))

Total number of unique words: 1130


#### 代码清单2-2 创建词汇表

In [7]:
vocab = {token:integer for integer, token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


#### 代码清单2-3 实现简单的文本分词器

In [8]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        # 字符串到整数的映射
        self.str_to_int = vocab  # 将词汇表作为类属性存储，以便在 encode 方法和 decode 方法中访问

        # 整数到字符串的映射（用于解码）
        self.int_to_str = {i: s for s, i in vocab.items()}

    # 文本转换为整数ID
    def encode(self, text):
        # 使用正则表达式分割文本，保留分隔符
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        # 移除空白字符并清理预处理后的标记
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]

        # 将标记转换为对应的整数ID
        ids = [self.str_to_int[token] for token in preprocessed]
        return ids

    # 整数ID转换为文本
    def decode(self, ids):
        # 将整数ID转换为对应的文字，并用空格连接
        text = " ".join([self.int_to_str[i] for i in ids])

        # 移除标点符号前的空格，使文本更自然
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

##### 测试分词器

In [9]:
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know,"Mrs.Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [10]:
print(tokenizer.decode(ids))

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [11]:
# 如果有训练集之外的新样本，则会报错 KeyError: 'Hello'
text = "Hello, do you like tea?"
print(tokenizer.encode(text))

KeyError: 'Hello'

##### 将 <|unk|> 和 <|endoftest|> 词元添加到词汇表中

In [17]:
all_tokens = sorted(set(preprocessed))
all_tokens.extend(["<|unk|>", "<|endoftext|>"])  # <|unk|> 词元用于表示未知单词，<|endoftext|> 词元用于分隔两个不相关的文本来源
vocab = {token:integer for integer, token in enumerate(all_tokens)}

print("Total number of unique words:", len(all_tokens))

Total number of unique words: 1132


#### 代码清单2-4 能够处理未知单词的文本分词器

In [18]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]

        # 用<|unk|>词元替换未知单词
        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>"
            for item in preprocessed
        ]

        ids = [self.str_to_int[token] for token in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

##### 测试分词器V2

In [19]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = "<|endoftext|> ".join([text1, text2])
print( text)

Hello, do you like tea?<|endoftext|> In the sunlit terraces of the palace.


In [20]:
tokenizer = SimpleTokenizerV2(vocab)
print(tokenizer.encode(text))

[1130, 5, 355, 1126, 628, 975, 10, 1131, 55, 988, 956, 984, 722, 988, 1130, 7]


In [21]:
print(tokenizer.decode(tokenizer.encode(text)))

<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


##### 基于 BPE 概念的更复杂的分词方案

In [22]:
# pip install tiktoken
from importlib.metadata import version
import tiktoken
print("tiktoken version:", version("tiktoken"))  # 检查tiktoken版本

tiktoken version: 0.9.0


In [23]:
# 实例化 tiktoken 中的 BPE 分词器
tokenizer = tiktoken.get_encoding("gpt2")

In [24]:
# 使用方法与 SimpleTokenizerV2 类似，使用 encode 方法将文本转换为整数ID，使用 decode 方法将整数ID转换为文本
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace."
)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 262, 20562, 13]


In [25]:
strings = tokenizer.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


##### BPE 算法的原理是将不在预定义的单词分解为更小的子词甚至单个字符，从而能够处理词汇表之外的单词

##### 实现一个数据加载器，使用滑动窗口方法从训练数据集中提取输入-目标对

In [26]:
# 使用 BPE 分词器进行分词
with open ("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print("Total number of tokens:", len(enc_text))

Total number of tokens: 5145


In [27]:
# 移除前50个词元以便演示
enc_sample = enc_text[50:]

In [29]:
context_size = 4  # 上下文大小决定的输入中包含了多少个词元
x = enc_sample[: context_size]  # 输入
y = enc_sample[1 : context_size + 1]  # 目标
print(f"x: {x}")
print(f"y: {y}")

x: [290, 4920, 2241, 287]
y: [4920, 2241, 287, 257]


In [31]:
# 创建预测任务
for i in range(1, context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(f"{context} ----> {desired}")

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [32]:
for i in range(1, context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


#### 代码清单2-5 一个用于批处理输入和目标的数据集

In [33]:
import torch
from torch.utils.data import Dataset

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # 对全部文本进行分词
        token_ids = tokenizer.encode(txt)
        # 使用滑动窗口将文本划分为长度为max_length的重叠序列
        for i in range(0, len(token_ids) - max_length, stride):
            input_ids = token_ids[i: i + max_length]
            target_ids = token_ids[i + 1: i + 1 + max_length]
            self.input_ids.append(torch.tensor(input_ids))
            self.target_ids.append(torch.tensor(target_ids))

    # 返回数据集的总行数
    def __len__(self):
        return len(self.input_ids)

    # 返回数据集的指定行
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

#### 代码清单2-6 用于批量生成输入-目标对的数据加载器

In [34]:
# batch_size: 批次大小
# max_length: 输入和目标序列的长度
# stride: 滑动窗口的步长
# shuffle: 是否打乱数据
# drop_last: 是否删除最后一个不完整的批次
# num_workers: 用于预处理的CPU进程数
def create_dataloader_V1(
        txt, batch_size = 4, max_length = 256, stride = 128, shuffle = True, drop_last = True, num_workers = 0
):
    # 初始化分词器
    tokenizer = tiktoken.get_encoding("gpt2")
    # 创建数据集
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = torch.utils.data.DataLoader(
        dataset,
        batch_size = batch_size,
        shuffle = shuffle,
        drop_last = drop_last,    # 如果drop_last为True且批次大小小于指定的batch_size，则会删除最后一批，以防止在训练期间出现损失剧增
        num_workers = num_workers,    # 用于预处理的CPU进程数
    )
    return dataloader

In [35]:
# 测试数据加载器
with open ("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader = create_dataloader_V1(
    raw_text, batch_size = 1, max_length = 4, stride = 1, shuffle = False
)

data_iter = iter(dataloader)  # 将 dataloader 转换为 Python 迭代器,以通过 Python 内置的 next() 函数获取下一个条目
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [36]:
# 第二批数据内容
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [37]:
# 以大于 1 的批次大小使用数据加载器进行采样
dataloader = create_dataloader_V1(
    raw_text, batch_size = 8, max_length = 4, stride = 4, shuffle = False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("Targets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


##### 创建词元嵌入

In [38]:
input_ids = torch.tensor([2, 3, 5, 1]) # 示例输入ID序列

In [39]:
vocab_size = 6 # 词汇表大小
output_dim = 3 # 词向量的输出维度

In [40]:
# 将随机种子设置为123
torch.manual_seed(123)
# 主要组件说明
# torch.nn.Embedding
# 这是PyTorch中的嵌入层类
# 用于将离散的索引（如词汇ID）映射为连续的向量表示
# 参数含义
# vocab_size: 词汇表大小，表示有多少个不同的词或token
# output_dim: 输出维度，即每个词被映射成多少维的向量
# 功能作用
# 词嵌入转换: 将输入的整数索引转换为密集向量
# 可学习参数: 嵌入层内部维护一个形状为 (vocab_size, output_dim) 的权重矩阵
# 前向传播: 输入索引时，自动查找对应行的向量作为输出
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [41]:
print(embedding_layer(torch.tensor([3]))) # 索引3对应的词向量，即第四行

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


In [42]:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


##### 编码单词位置信息

In [43]:
# 创建一个嵌入层
vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [44]:
# 创建数据加载器
max_length = 4
dataloader = create_dataloader_V1(
    raw_text, batch_size = 8, max_length = max_length, stride = max_length, shuffle = False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Token IDs:\n", inputs)
print("\nInput shape:", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Input shape: torch.Size([8, 4])


In [45]:
token_embeddings = token_embedding_layer(inputs)  # 嵌入单词
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [46]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [47]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])
